In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding


# 1. Initialize your local model
local_embed = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# 2. Build storage context with your explicit model
storage_context = StorageContext.from_defaults()


/home/chins/git/rag-architecture/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 901.45it/s]


In [ ]:
# # 3. Pass it directly into the index function
# documents = SimpleDirectoryReader(
#     input_dir="../staging/",
#     file_extractor=file_extractor
# ).load_data()

# index = VectorStoreIndex.from_documents(
#     documents, 
#     embed_model=local_embed, # Passed explicitly
#     storage_context=storage_context
# )

# query_engine = index.as_query_engine()
# retriever = index.as_retriever()

In [ ]:

async def search_documents(query: str) -> str:
    """Useful for answering natural language questions about individuals"""
    response = await query_engine.aquery(query)
    return str(response)


# Create an enhanced workflow with both tools
agent = FunctionAgent(
    tools=[search_documents],
    llm=OpenAI(model="gpt-4o-mini"),
    system_prompt="""You are a helpful assistant that can search through documents to answer questions. 
    Do not answer questions that are not related to the documents. If you cannot find the answer in the documents, say "I don't know".""",
)

# Now we can ask questions about the documents or do calculations
async def main():
    response = await agent.run(
        "What are the differences between the play and the drama?"
    )
    print(response)

await main()

In [ ]:
from dataclasses import dataclass
from markdown_it import MarkdownIt
from bisect import bisect_left, bisect_right
from collections import defaultdict

@dataclass
class SectionMetadata:
    section_id: str = ''
    section_title: str = ''
    section_start_char_idx: int = 0 #index in pdf document text where the section starts
    section_end_char_idx: int = 0 #index in pdf document text where the section ends
    
    pdf_file_name: str = '' 
    pdf_file_path: str = ''
    pdf_title: str = ''
    pdf_author: str = ''
    pdf_length: int = 0 #no of characters in pdf file
    pdf_total_pages: int = 0 #total pages in the parent document
    pdf_page_offsets: list = None #pdf page offset positions w.r.t characters 
    document_id: str = ''
    
@dataclass
class MarkdownSection:
    level: int
    text: str
    
    metadata: SectionMetadata

@dataclass
class MarkdownChunk:
    text: str

    chunk_id: str = ''
    start_char_idx: int = 0 #start index of chunk in the original pdf file character stream
    end_char_idx: int = 0 #end index of chunk in the original pdf file character stream
    start_page_idx: int = 0 #pdf page number index where the chunk starts
    end_page_idx: int = 0 #pdf page number index where the chunk ends

    section: MarkdownSection = None

'''
returns start character indexes for each page (0-based). 
'''
def build_page_offsets(pages):
    offsets = []
    start = 0

    for i, page in enumerate(pages):
        offsets.append(start)
        start = start + len(page['text'])
    return offsets

'''
returns pdf file level metadata and page_offsets (mapping between pages and their start character positions)
'''
def get_pdf_metadata(pages):
    metadata = defaultdict(None)
    keys = ['title', 'author', 'file_path', 'page_count']
    if pages and 'metadata' in pages[0]:
        obj = pages[0]['metadata']
        metadata = defaultdict(None, {key : obj[key] for key in keys if key in obj})

    page_offsets = build_page_offsets(pages)
    return metadata, page_offsets

'''
returns index for each new line character (0 based) 
Note that if there are 'm' newline characters (\n), then we have m+1 lines.
so offsets[i] = character index of ith line (0 based) 
'''
def build_line_offsets(text):
    
    offsets = [0] #first line always start at 0th index

    for i, char in enumerate(text):
        if char == "\n":
            offsets.append(i)

    return offsets

'''
takes markdown and and returns the list of heirarchical sections
'''
def markdown_sections(markdown, page_offsets, file_metadata):

    md = MarkdownIt("commonmark")
    tokens = md.parse(markdown)
    line_offsets = build_line_offsets(markdown)
    headings = []

    # pass 1: extract each heading (often heirarchical), its level and start line
    for i, token in enumerate(tokens):

        if token.type != "heading_open":
            continue

        level = int(token.tag[1]) #e.g for h1, level=1

        # <heading_open> <inline> <heading_close>
        inline_token = tokens[i + 1]
        section_title = inline_token.content
        start_line = token.map[0]

        headings.append({
            "level": level,
            "section_title": section_title,
            "start_line": start_line, 
        })

    # pass 2: create sections from the headings. A section defined by a heading at level l ends when we encounter the 
    # first heading with level <= l (or the end of text)
    # note that start_line, start_idx are inclusive, end_line, end_idx are exclusive
    sections = []
    n = len(headings)

    for i in range(n): #for each heading index
        heading = headings[i]
        level = heading["level"]
        start_line = heading["start_line"]

        # Find the next heading with the same or lower level. That ends the current section
        end_line = len(line_offsets)
        for j in range(i+1, n):
            next_heading = headings[j]
            if next_heading["level"] <= level: #new section start detected. it's start is the end of the current section
                end_line = next_heading["start_line"]
                break

        #map the section start and end lines to start and end character positions 
        start_idx = line_offsets[start_line]

        if end_line < len(line_offsets):
            end_idx = line_offsets[end_line]
        else:
            end_idx = len(markdown)

        sections.append(
            MarkdownSection(
                level=level,
                text=markdown[start_idx:end_idx],

                metadata=SectionMetadata(
                    section_title=heading["section_title"],
                    section_start_char_idx = start_idx,
                    section_end_char_idx = end_idx,

                    pdf_title=file_metadata.get('title', ''), 
                    pdf_author=file_metadata.get('author', ''), 
                    pdf_file_path=file_metadata.get('file_path', ''), 
                    pdf_total_pages=file_metadata.get('page_count', ''), 
                    pdf_page_offsets=page_offsets
                    
                    )
            )
        )
        
    return sections

'''
udpates the page_start and page_end attributes of section based on page_offset 
'''
def update_page_start_and_end(section, page_offsets):
    if section:
        section.metadata.start_page = bisect_right(page_offsets, section.start_pos) - 1
        section.metadata.end_page = bisect_left(page_offsets, section.end_pos) - 1

'''
Takes the hierarchical sections and the cut_level and return a list of non-overlapping and collectively exhausting flat sections.
All sections at level > cut_level are simply absorbed in their parent sections.
For all the sections at level < cut_level (which are parent sections), their spans are adjusted so that they end right where
the first child under them begins resulting in non-overlapping sections.
'''
def get_flat_sections(sections, page_offsets, metadata, cut_level=2): 

    root = sections[0]
    markdown = root.text

    #remove (i.e merge with their parent) all sections at finer level than the cut level 
    flat_sections = [section for section in sections if section.level <= cut_level]

    #update end range and text for each parent section
    n = len(flat_sections)

    #process sections from end to start. set start_page and end_page for the last section
    next_section = flat_sections[-1]
    #update_page_start_and_end(next_section, page_offsets)

    #process from second last to first sections
    for i in range(n-1)[::-1]: 
        curr_section = flat_sections[i]

        #if curr section is parent of next section, adjust current section's span so that it ends where the next section begins
        if curr_section.level < next_section.level: 

            curr_section.metadata.section_end_char_idx = next_section.metadata.section_start_char_idx

            #if this is the first section, just take everything from the start
            #this is to avoid extra \n at the beginning of the document
            if i == 0: 
                curr_section.text = markdown[: curr_section.metadata.section_end_char_idx]
            else:
                curr_section.text = markdown[curr_section.metadata.section_start_char_idx: curr_section.metadata.section_end_char_idx]

        #update start and end page for current section
        #update_page_start_and_end(curr_section, page_offsets)

        #set curr as the next section for the next iteration
        next_section = curr_section
        
    return flat_sections

In [4]:
import pymupdf4llm
from markdown_it import MarkdownIt
from llama_index.core import Document

#read full_text as well as
pages = pymupdf4llm.to_markdown('../staging/2505.07891.pdf', page_chunks=True)
full_md = ''.join([page['text'] for page in pages])


In [5]:
file_metadata, page_offsets = get_pdf_metadata(pages)

#parse the markdown to find logical sections (treating h2 as the cut_level)
hierarchical_sections = markdown_sections(full_md, page_offsets, file_metadata)
sections = get_flat_sections(hierarchical_sections, page_offsets, file_metadata)


In [ ]:
from dataclasses import asdict
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import NodeRelationship

'''
Returns chunk object for a given llamaindex chunk
'''
def get_chunk(node):

    source_doc = node.relationships[NodeRelationship.SOURCE]
    parent_section = source_doc.metadata
    page_offsets = parent_section['pdf_page_offsets']
    section_start_idx = parent_section['section_start_char_idx'] 

    chunk_start_idx = section_start_idx + node.start_char_idx
    chunk_end_idx = chunk_start_idx + len(node.text) 

    chunk_start_page_idx = bisect_right(page_offsets, chunk_start_idx) - 1
    chunk_end_page_idx = bisect_left(page_offsets, chunk_end_idx) - 1
     
    return MarkdownChunk(
            text = node.text,

            chunk_id = node.id_,
            start_char_idx = chunk_start_idx,
            end_char_idx= chunk_end_idx,
            start_page_idx=chunk_start_page_idx,
            end_page_idx=chunk_end_page_idx,

            section = parent_section,
            
            )

def update_node_metadata(node, chunk):

    node.metadata = {
        **node.metadata,
        'chunk_id': node.id_,
        'chunk_start_char_idx': chunk.start_char_idx,
        'chunk_end_char_idx': chunk.end_char_idx,
        'chunk_start_page_idx': chunk.start_page_idx,
        'chunk_end_page_idx': chunk.end_page_idx,
        }
    
#build documents corresponding to each section
documents = [Document(text=section.text, metadata=asdict(section.metadata)) for section in sections]

#create chunks (nodes)
splitter = SentenceSplitter(
    chunk_size=1024,
    chunk_overlap=128,
)
nodes = splitter.get_nodes_from_documents(documents)

#build chunk objects for citation support
chunks = []
for node in nodes:
    chunk = get_chunk(node)
    update_node_metadata(node, chunk)
    chunks.append(chunk)

#build index over nodes
index = VectorStoreIndex(
    nodes, 
    embed_model=local_embed, 
    storage_context=storage_context
    )



In [ ]:
# for chunk in chunks:
#     print(f'---------------------------------------------------------')
    
#     print(f'chunk span:{chunk.chunk_start_char_idx}-{chunk.chunk_end_char_idx}, pages:{chunk.chunk_start_page}-{chunk.chunk_end_page}')
#     print(f'parent section span:{chunk.section['section_start_char_idx']}-{chunk.section['section_end_char_idx']}')


In [ ]:
from llama_index.core.schema import NodeRelationship
node_idx = 10

chunk = nodes[node_idx]
source_doc = chunk.relationships[NodeRelationship.SOURCE]
chunk_start_idx, chunk_end_idx = chunk.start_char_idx, chunk.end_char_idx

print(source_doc.metadata['start_page'])
print(chunk_start_idx, chunk_end_idx)

In [67]:
from pydantic import BaseModel, Field

class Citation(BaseModel):
    ''' A citation source'''
    id: int = Field(description="source #")

class AnswerPart(BaseModel):
    '''A part of the answer backed by citation'''
    part: str =  Field(description="The answer part text")
    citations: list[Citation] = Field(description="A list of citations backing the answer part")

class RAGResponse(BaseModel):
    '''Citation backed answer. An Answer consists of one or more parts'''
    parts: list[AnswerPart] = Field(description="A list of citations backing the answer")


In [75]:
from llama_index.core.query_engine import CitationQueryEngine
from llama_index.core import PromptTemplate

custom_citation_prompt = PromptTemplate(
    "Please answer the query based on the provided context.\n"
    "Every time you use a source text to generate your answer part, you must add the corresponding citation source.\n"
    
    "Do not add explicit references in square brackets in your answer. Instead use the citations in the structured output resposne."
    "Do not use external knowledge. If you do not know the answer, send empty response.\n\n"
    "Context:\n{context_str}\n\n"
    "Query: {query_str}\n\n"
    "Answer:"
)

llm = OpenAI(model="gpt-4o-mini").as_structured_llm(RAGResponse)

query_engine = CitationQueryEngine.from_args(
    index,
    llm=llm,
    citation_chunk_size=512,       # Granular text blocks for inline citations
    citation_chunk_overlap=20,     # Small overlap to prevent cut-off quotes
    similarity_top_k=3,            # Number of source nodes to retrieve
    citation_qa_template=custom_citation_prompt,
)

retriever = index.as_retriever()

In [69]:
response = query_engine.query('Explain the topic specific text-rank in short. Also, what is the benefit of TrumourGPT?')
response

PydanticResponse(response=RAGResponse(parts=[AnswerPart(part='Topic-specific TextRank (TST) is an adaptation of the traditional TextRank algorithm that prioritizes sentences based on their relevance to a specific topic. This approach enhances sensitivity to thematic elements by modifying vertex selection, edge weighting, and the scoring algorithm to emphasize topic relevance. For instance, in health-related contexts, sentences associated with health topics are assigned higher weights, allowing TST to effectively identify and rank sentences that are most pertinent to the target topic.', citations=[Citation(id=1)]), AnswerPart(part="TrumorGPT leverages the capabilities of Generative Pre-trained Transformer 4 (GPT-4) to construct accurate semantic health knowledge graphs. By utilizing algorithms like topic-enhanced sentence centrality and topic-specific TextRank, TrumorGPT can extract key phrases and sentences that represent central ideas within the text. This method allows for efficient 

PydanticResponse(response=RAGResponse(parts=[]), source_nodes=[NodeWithScore(node=TextNode(id_='60912250-c934-4467-a858-8bb55cc19fca', embedding=None, metadata={'section_id': '', 'section_title': 'III. METHODS', 'section_start_char_idx': 17688, 'section_end_char_idx': 18939, 'pdf_file_name': '', 'pdf_file_path': '../staging/2505.07891.pdf', 'pdf_title': 'TrumorGPT: Graph-Based Retrieval-Augmented Large Language Model for Fact-Checking', 'pdf_author': 'Ching Nam Hang; Pei-Duo Yu; Chee Wei Tan', 'pdf_length': 0, 'pdf_total_pages': 15, 'pdf_page_offsets': [0, 6438, 12953, 19557, 22823, 28281, 33935, 39954, 44496, 50779, 57103, 61234, 65007, 69930, 77919], 'document_id': '', 'chunk_id': 'dcfe06e2-5d1d-49f2-ba03-62987d8b0466', 'chunk_start_char_idx': 17689, 'chunk_end_char_idx': 18937, 'chunk_start_page_idx': 2, 'chunk_end_page_idx': 2}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='d8f6cb97-7f2c-416d-

In [ ]:
# from llama_index.core.schema import NodeRelationship
# sources = response.source_nodes

# text1 = sources[0].node.text[10:]
# text2 = sources[1].node.text[10:]

# source_chunk_id = sources[0].metadata['chunk_id']
# source_node = index.docstore.get_node(source_chunk_id)

# print(f'node text: {source_node.text[:100]}......{source_node.text[-100:]}')

# print(f'text1 start: {text1[:100]}')
# print(f'text2 end  : {text2[-100:]}')

# len(text1 + text2), len(source_node.text)


node text: ## _C. Topic-Specific TextRank_ 

Topic-specific TextRank (TST) adapts the traditional TextRank algo...... From the definition of _pi,j_ in (2), we have 



where the results above follow from the fact that
text1 start: ## _C. Topic-Specific TextRank_ 

Topic-specific TextRank (TST) adapts the traditional TextRank algo
text2 end  : From the definition of _pi,j_ in (2), we have 



where the results above follow from the fact that



(2964, 2957)

In [ ]:
async def search_documents(query: str) -> str:
    """Useful for answering natural language questions about documents"""
    response = await query_engine.aquery(query)
    return str(response)


# Create an enhanced workflow with both tools
agent = FunctionAgent(
    tools=[search_documents],
    llm=OpenAI(model="gpt-4o-mini"),
    system_prompt="""
    You are a helpful assistant that can search through documents to answer questions. 
    Do not answer questions that are not related to the documents. For provenance, generate citation which includes the details about the source file name, section name, and the psource age(s) 
    which your answers are based on. If you cannot find the answer in the documents, say "I don't know".""",
)

# Now we can ask questions about the documents or do calculations
async def main():
    response = await agent.run(
        "Explain the topic specific text-rank. Also, what are the three techniques used by Trurumour to verify the truthfulness of the health news?"
    )
    print(response)

await main()

In [ ]:

async def search_documents(query: str) -> str:
    """Useful for answering natural language questions about individuals"""
    response = await query_engine.aquery(query)
    return str(response)


# Create an enhanced workflow with both tools
agent = FunctionAgent(
    tools=[search_documents],
    llm=OpenAI(model="gpt-4o-mini"),
    system_prompt="""You are a helpful assistant that can search through documents to answer questions. 
    Do not answer questions that are not related to the documents. If you cannot find the answer in the documents, say "I don't know".""",
)

# Now we can ask questions about the documents or do calculations
async def main():
    response = await agent.run(
        "What is Trumor GPT?"
    )
    print(response)

await main()